# Experiment 02 — Reconstructing missing pixels

**Question:** What does a model learn when its job is to predict the exact RGB values of a hidden region?

This is our baseline. Later, JEPA will see the same kind of hidden region but predict its embedding instead of its pixels.

## Our prediction

We expect the loss to fall and the model to learn plausible colours and textures. Exact reconstruction may remain blurry because the visible region does not uniquely determine every hidden pixel.

In [ ]:
from pathlib import Path
import sys

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the JEPA Experiments project root')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset

from jepa.data import load_cifar10
from jepa.device import best_device
from jepa.masking import centered_block_mask, paint_masked_patches
from jepa.models import PatchAutoencoder, masked_pixel_loss
from jepa.patches import patchify, unpatchify

torch.manual_seed(7)
device = best_device()
device

## 1. Fix the experimental configuration

We use 10,000 training images for a quick first run. The model never receives the CIFAR-10 labels.

In [ ]:
IMAGE_SIZE = 32
PATCH_SIZE = 4
GRID_SIZE = IMAGE_SIZE // PATCH_SIZE
EMBEDDING_DIM = 128
BATCH_SIZE = 64
TRAINING_IMAGES = 10_000
EPOCHS = 10
LEARNING_RATE = 3e-4

In [ ]:
dataset = load_cifar10(train=True)
training_data = Subset(dataset, range(TRAINING_IMAGES))
loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
images, _ = next(iter(loader))
print('Batch shape:', tuple(images.shape))

## 2. Hide the centre

The image contains an 8 × 8 patch grid. We hide a 4 × 4 block in its centre: 16 of the 64 patches, or 25% of the image area.

In [ ]:
mask = centered_block_mask(GRID_SIZE, block_size=4, device=device)
visible_images = paint_masked_patches(images[:6], mask.cpu(), patch_size=PATCH_SIZE)

figure, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for column in range(6):
    axes[0, column].imshow(images[column].permute(1, 2, 0))
    axes[1, column].imshow(visible_images[column].permute(1, 2, 0))
    axes[0, column].axis('off')
    axes[1, column].axis('off')
axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('Visible')
figure.suptitle('The model must predict the grey region')
figure.tight_layout();

## 3. Build the pixel-prediction model

Each patch becomes a 128-number token. Hidden patches receive the same learned mask token. Position embeddings tell the transformer where every patch belongs. The final head predicts 48 RGB values for each patch.

In [ ]:
model = PatchAutoencoder(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    embedding_dim=EMBEDDING_DIM,
    depth=4,
    heads=4,
).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'Parameters: {parameter_count:,}')

In [ ]:
predicted_patches, target_patches = model(images[:2].to(device), mask)
print('Target patches:   ', tuple(target_patches.shape))
print('Predicted patches:', tuple(predicted_patches.shape))
print('Hidden target:    ', tuple(target_patches[:, mask].shape))

## 4. Measure error only where the image is hidden

The model predicts every patch, but the loss uses only the 16 hidden ones. For each hidden patch, it compares all 48 predicted channel values with the real values using mean squared error.

In [ ]:
initial_loss = masked_pixel_loss(predicted_patches, target_patches, mask)
print('Loss before training:', initial_loss.item())

## 5. Train

Every update asks the same question: how should the model change so its hidden RGB values become closer to the real ones?

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    examples_seen = 0

    for batch, _ in loader:
        batch = batch.to(device)
        predicted, target = model(batch, mask)
        loss = masked_pixel_loss(predicted, target, mask)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.shape[0]
        examples_seen += batch.shape[0]

    epoch_loss = total_loss / examples_seen
    losses.append(epoch_loss)
    print(f'Epoch {epoch + 1:02d}/{EPOCHS}: {epoch_loss:.6f}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), losses, marker='o', color='#7c3aed')
plt.xlabel('Epoch')
plt.ylabel('Masked pixel MSE')
plt.title('Can the model predict the hidden pixels?')
plt.grid(alpha=0.25);

## 6. Look at what it predicted

A loss number tells us whether predictions moved closer to the target. The images show what that improvement actually means.

In [ ]:
model.eval()
examples = images[:6]
with torch.no_grad():
    predicted, _ = model(examples.to(device), mask)

predicted_images = unpatchify(
    predicted.cpu(), patch_size=PATCH_SIZE, image_size=IMAGE_SIZE
).clamp(0, 1)
completed = examples.clone()
for row, column in mask.cpu().reshape(GRID_SIZE, GRID_SIZE).nonzero(as_tuple=False):
    y, x = int(row) * PATCH_SIZE, int(column) * PATCH_SIZE
    completed[:, :, y:y + PATCH_SIZE, x:x + PATCH_SIZE] = (
        predicted_images[:, :, y:y + PATCH_SIZE, x:x + PATCH_SIZE]
    )

figure, axes = plt.subplots(3, 6, figsize=(12, 6.5))
for column in range(6):
    for row, collection in enumerate((examples, visible_images, completed)):
        axes[row, column].imshow(collection[column].permute(1, 2, 0).clamp(0, 1))
        axes[row, column].axis('off')
axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('Visible')
axes[2, 0].set_ylabel('Prediction')
figure.suptitle('What the model placed in the hidden region')
figure.tight_layout();

## What to record

Before moving to JEPA, record the final loss, whether the output is sharp or blurry, whether colours are plausible, and which kinds of details the model fails to recover.

The important point is the target: this model is rewarded for predicting every hidden RGB value. Experiment 03 will change that target from pixels to embeddings.